## Hy-MMSBM more then two unique years and at least 2 chronic groups

- Goal: Identify the overlapping higher-order organisation of chronic diseases from complete patient multimorbidity profiles using Hy-MMSBM.
- RQ1: Across the adult population, how are chronic diseases organized into overlapping communities when complete multimorbidity profiles are considered?
- Target: the pooled population-level higher-order disease organization//not age or sex stratified different models!

Limitation of HY - MMSBM: possible demographic influence on pooled clustering

Because Hy-MMSBM is fitted on the pooled population, some detected communities may be influenced by age- or sex-specific multimorbidity patterns.

* Literature: pooled clustering is common. Haug et al. (2020) used one pooled DIVCLUS-T solution, and Ferris et al. (2025, DOI: 10.1038/s41467-025-67372-6) found that most multimorbidity clustering studies pooled age and sex.
* Our design: before Hy-MMSBM, age-specific hypergraphs are compared using hypergraph similarity to assess how strongly multimorbidity structure varies with age.
* Main goal: the pooled Hy-MMSBM is interpreted as the overall population-level disease-community structure, not as proof that the structure is identical across all demographic groups.

Load HYMMSBM

In [11]:
import sys
import subprocess
from pathlib import Path
import pandas as pd

if sys.version_info[:2] != (3, 9):
    raise RuntimeError("Select the CNHypergraph Python 3.9 kernel.")

hy_python = Path(sys.executable)
HY_MMSBM_DIR = Path("third_party") / "Hy-MMSBM"
hy_inference_script = HY_MMSBM_DIR / "main_inference.py"

## 1. Load and select the primary cohort

Observed years come from eligible stays in `patient_table.csv`. The membership `calendar_year` is the year in which a chronic group was first recorded and is not used as a follow-up measure.

In [12]:
chronic_memberships_file = Path("chronic_memberships.csv")
patient_table_file = Path("patient_table.csv")
chronic_mapping_file = Path("chronic_mapping.csv")
block_mapping_file = Path("block_mapping.csv")

chronic_memberships = pd.read_csv(
    chronic_memberships_file,
    parse_dates=["first_date"],
)
patient_table = pd.read_csv(
    patient_table_file,
    parse_dates=["first_stay_date", "last_stay_date", "death_date"],
)


In [13]:
primary_cohort_mask = (
    patient_table["n_unique_years"].ge(2)
    & patient_table["n_chronic_groups"].ge(2)
)
hy_patient_table = (
    patient_table.loc[primary_cohort_mask]
    .copy()
    .sort_values("patient_no")
    .reset_index(drop=True)
)
hy_patient_ids = set(hy_patient_table["patient_no"])

hy_chronic_memberships = (
    chronic_memberships.loc[
        chronic_memberships["patient_no"].isin(hy_patient_ids)
    ]
    .sort_values(["patient_no", "first_date", "node_key"])
    .reset_index(drop=True)
)

# 2. Aggregating patient profiles

In [14]:
# Assign the 34 chronic groups deterministic integer node IDs.
chronic_nodes = (
    hy_chronic_memberships[["node_key", "node_label"]]
    .drop_duplicates()
    .sort_values("node_key")
    .reset_index(drop=True)
)

chronic_nodes["node_id"] = range(len(chronic_nodes))

membership_with_node_id = hy_chronic_memberships.merge(
    chronic_nodes[["node_key", "node_id"]],
    on="node_key",
)

In [15]:
# one sorted disease profile per patient
patient_profiles = (
    membership_with_node_id
    .groupby("patient_no")["node_id"]
    .agg(lambda values: tuple(sorted(set(values))))
    .rename("hyperedge")
)
# Compress identical profiles
profile_counts = patient_profiles.value_counts()

weighted_profiles = pd.DataFrame(
    sorted(
        profile_counts.items(),
        key=lambda item: (len(item[0]), item[0]),
    ),
    columns=["hyperedge", "weight"],
)

## INPUT AUDIT

inspect how many unique profiles remain after compression // tells how large the final Hy-MMSBM input will be.

In [16]:
# the exact input format expected by Hy-MMSBM

HY_INPUT_DIR = Path("hy_input")
HY_INPUT_DIR.mkdir(exist_ok=True)

hyperedge_file = HY_INPUT_DIR / "hyperedges.txt"
weight_file = HY_INPUT_DIR / "weights.txt"
audit_summary_file = HY_INPUT_DIR / "inputAuditChronicDisease.csv"
hyperedge_size_file = HY_INPUT_DIR / "hyperedgeSizeDistributionChronicDisease.csv"
prevalence_file = HY_INPUT_DIR / "prevalenceChronicDisease.csv"
hyperedge_size_plot_file = HY_INPUT_DIR / "hyperedgeSizeDistributionChronicDisease.png"
prevalence_plot_file = HY_INPUT_DIR / "prevalenceChronicDisease.png"
profile_weight_plot_file = HY_INPUT_DIR / "profileWeightDistributionChronicDisease.png"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

n_patients = int(weighted_profiles["weight"].sum())

input_audit_summary = pd.DataFrame({
    "patients": [n_patients],
    "unique_profiles": [len(weighted_profiles)],
    "active_disease_nodes": [membership_with_node_id["node_id"].nunique()],
})

profile_audit = weighted_profiles.assign(
    hyperedge_size=weighted_profiles["hyperedge"].map(len)
)
hyperedge_size_distribution = (
    profile_audit.groupby("hyperedge_size", as_index=False)
    .agg(
        unique_profiles=("hyperedge", "size"),
        patients=("weight", "sum"),
    )
)

disease_frequency = (
    membership_with_node_id.groupby("node_id", as_index=False)
    .agg(patients=("patient_no", "nunique"))
    .merge(chronic_nodes, on="node_id", how="left")
    .assign(prevalence=lambda table: 100 * table["patients"] / n_patients)
)
top_prevalent_diseases = (
    disease_frequency.nlargest(10, "prevalence")
    .sort_values("prevalence")
)
disease_prevalence = (
    disease_frequency.rename(
        columns={
            "patients": "frequency",
            "prevalence": "prevalence_percentage",
        }
    )
    [["node_id", "node_key", "node_label", "frequency", "prevalence_percentage"]]
    .assign(
        prevalence_percentage=lambda table: table["prevalence_percentage"].round(3)
    )
    .sort_values("prevalence_percentage", ascending=False)
)
weight_groups = pd.cut(
    weighted_profiles["weight"],
    bins=[0, 1, 5, 10, float("inf")],
    labels=["1", "2–5", "6–10", ">10"],
).value_counts(sort=False)

input_audit_summary.to_csv(audit_summary_file, index=False)
hyperedge_size_distribution.to_csv(hyperedge_size_file, index=False)
disease_prevalence.to_csv(prevalence_file, index=False)

figure, axis = plt.subplots(figsize=(8, 5))
positions = list(range(len(hyperedge_size_distribution)))
axis.bar(
    [position - 0.2 for position in positions],
    hyperedge_size_distribution["unique_profiles"],
    width=0.4,
    label="Unique profiles",
)
axis.bar(
    [position + 0.2 for position in positions],
    hyperedge_size_distribution["patients"],
    width=0.4,
    label="Patients",
)
axis.set_xticks(positions, hyperedge_size_distribution["hyperedge_size"])
axis.set(
    title="Hyperedge-size distribution",
    xlabel="Chronic groups per profile",
    ylabel="Count",
)
axis.legend()
figure.tight_layout()
figure.savefig(hyperedge_size_plot_file, dpi=300, bbox_inches="tight")
plt.close(figure)

figure, axis = plt.subplots(figsize=(9, 6))
prevalence_bars = axis.barh(
    top_prevalent_diseases["node_label"],
    top_prevalent_diseases["prevalence"],
)
axis.bar_label(
    prevalence_bars,
    labels=[
        f"{prevalence:.1f}% (n={patients:,})"
        for prevalence, patients in zip(
            top_prevalent_diseases["prevalence"],
            top_prevalent_diseases["patients"],
        )
    ],
    padding=3,
)
axis.set(
    title="Top 10 chronic diseases by recorded prevalence",
    xlabel="Selected patients (%)",
    ylabel="Chronic disease group",
)
axis.set_xlim(0, top_prevalent_diseases["prevalence"].max() * 1.3)
figure.tight_layout()
figure.savefig(prevalence_plot_file, dpi=300, bbox_inches="tight")
plt.close(figure)

figure, axis = plt.subplots(figsize=(7, 5))
weight_groups.plot.bar(ax=axis, color="steelblue")
axis.set(
    title="Profile-weight distribution",
    xlabel="Patients sharing the same profile",
    ylabel="Unique profiles",
)
axis.bar_label(axis.containers[0])
axis.tick_params(axis="x", rotation=0)
figure.tight_layout()
figure.savefig(profile_weight_plot_file, dpi=300, bbox_inches="tight")
plt.close(figure)


Saving weight and hyperedges files as txt-input to model

In [17]:
# a file containing the distinct hyperedges-tuples so intermediate conversion avoiding
with hyperedge_file.open("w", encoding="utf-8") as file:
    for hyperedge in weighted_profiles["hyperedge"]:
        file.write(" ".join(map(str, hyperedge)) + "\n")

# a second file containing hyperedge weights
weighted_profiles["weight"].to_csv(
    weight_file,
    index=False,
    header=False,
)
#chronic_nodes!

# 3. Select K by repeated 80/20 hyperedge-prediction AUC

The selection rule is `selected_k = argmax(mean test AUC)`. The same 100 splits and same-size unobserved hyperedges are reused for every candidate K.

In [18]:
from itertools import chain
from tempfile import TemporaryDirectory

import numpy as np

candidate_k = range(2, 16)
n_cv_splits = 10#change to 100 for final runs
test_fraction = 0.20
training_rounds = 10
em_rounds = 100
base_seed = 42

HY_RESULTS_DIR = Path("hy_results")
HY_RESULTS_DIR.mkdir(exist_ok=True)

cv_results_file = HY_RESULTS_DIR / "aucBySplitChronicDisease.csv"
k_selection_file = HY_RESULTS_DIR / "kSelectionChronicDisease.csv"
k_selection_plot_file = HY_RESULTS_DIR / "kSelectionChronicDisease.png"
chronic_disease_blocks_file = (
    HY_RESULTS_DIR / "chronicDiseaseBlockMapping.csv"
)

def write_hyperedges(hyperedges, path):
    with path.open("w", encoding="utf-8") as file:
        for hyperedge in hyperedges:
            file.write(" ".join(map(str, hyperedge)) + "\n")


observed_hyperedges = set(weighted_profiles["hyperedge"])
active_node_ids = np.array(sorted(membership_with_node_id["node_id"].unique()))
n_test_profiles = round(test_fraction * len(weighted_profiles))
cv_splits = []

for split_number in range(1, n_cv_splits + 1):
    split_seed = base_seed + split_number
    negative_seed = base_seed + 10_000 + split_number
    split_rng = np.random.default_rng(split_seed)

    while True:
        shuffled_indices = split_rng.permutation(len(weighted_profiles))
        test_indices = shuffled_indices[:n_test_profiles]
        train_indices = shuffled_indices[n_test_profiles:]
        train_profiles = weighted_profiles.iloc[train_indices]
        train_nodes = set(chain.from_iterable(train_profiles["hyperedge"]))
        if train_nodes == set(active_node_ids):
            break

    test_profiles = weighted_profiles.iloc[test_indices]
    negative_rng = np.random.default_rng(negative_seed)
    negative_hyperedges = []
    for positive_hyperedge in test_profiles["hyperedge"]:
        while True:
            negative_hyperedge = tuple(sorted(negative_rng.choice(
                active_node_ids,
                size=len(positive_hyperedge),
                replace=False,
            )))
            if negative_hyperedge not in observed_hyperedges:
                negative_hyperedges.append(negative_hyperedge)
                break

    cv_splits.append({
        "split": split_number,
        "split_seed": split_seed,
        "negative_seed": negative_seed,
        "train_hyperedges": train_profiles["hyperedge"].tolist(),
        "train_weights": train_profiles["weight"].tolist(),
        "positive_hyperedges": test_profiles["hyperedge"].tolist(),
        "negative_hyperedges": negative_hyperedges,
    })


## Fit each K-split combination and calculate held-out AUC

For a fixed K and split, `training_rounds=10` makes Hy-MMSBM retain the random initialization with the highest training likelihood. Test AUC is then calculated from that retained model.

In [19]:
def run_hy_inference(
    k,
    train_hyperedge_file,
    train_weight_file,
    output_dir,
    seed,
):
    command = [
        str(hy_python),
        str(hy_inference_script),
        "--K", str(k),
        "--assortative", "false",
        "--u_prior", "0",
        "--w_prior", "1",
        "--max_hye_size", "none",
        "--training_rounds", str(training_rounds),
        "--em_rounds", str(em_rounds),
        "--seed", str(seed),
        "--hyperedge_file", str(train_hyperedge_file),
        "--weight_file", str(train_weight_file),
        "--out_dir", str(output_dir),
    ]
    subprocess.run(command, check=True)


def hyperedge_score(hyperedge, U, W):
    memberships = U[np.asarray(hyperedge, dtype=int)]
    membership_sum = memberships.sum(axis=0)
    within_node_terms = np.einsum(
        "ik,kl,il->", memberships, W, memberships
    )
    return 0.5 * (membership_sum @ W @ membership_sum - within_node_terms)


def matched_hyperedge_auc(positive_hyperedges, negative_hyperedges, U, W):
    positive_scores = np.array([
        hyperedge_score(edge, U, W) for edge in positive_hyperedges
    ])
    negative_scores = np.array([
        hyperedge_score(edge, U, W) for edge in negative_hyperedges
    ])
    return float(np.mean(
        (positive_scores > negative_scores)
        + 0.5 * (positive_scores == negative_scores)
    ))


def evaluate_cv_run(k, split_info):
    split_number = split_info["split"]
    model_seed = base_seed + 1_000 * k + split_number

    with TemporaryDirectory() as temporary_directory:
        temporary_directory = Path(temporary_directory)
        train_hyperedge_file = temporary_directory / "hyperedges.txt"
        train_weight_file = temporary_directory / "weights.txt"
        output_dir = temporary_directory / "model"

        write_hyperedges(
            split_info["train_hyperedges"], train_hyperedge_file
        )
        pd.Series(split_info["train_weights"]).to_csv(
            train_weight_file, index=False, header=False
        )
        run_hy_inference(
            k=k,
            train_hyperedge_file=train_hyperedge_file,
            train_weight_file=train_weight_file,
            output_dir=output_dir,
            seed=model_seed,
        )
        U = np.loadtxt(output_dir / "inferred_u.txt")
        W = np.loadtxt(output_dir / "inferred_w.txt")
        auc = round(matched_hyperedge_auc(
            split_info["positive_hyperedges"],
            split_info["negative_hyperedges"],
            U,
            W,
        ), 3)

    return {
        "K": k,
        "split": split_number,
        "AUC": auc,
        "model_seed": model_seed,
        "split_seed": split_info["split_seed"],
        "negative_seed": split_info["negative_seed"],
        "training_rounds": training_rounds,
    }


def run_cv_grid(k_values, split_numbers):
    if cv_results_file.exists():
        results = pd.read_csv(cv_results_file)
    else:
        results = pd.DataFrame()

    completed = (
        set(zip(results["K"], results["split"]))
        if not results.empty
        else set()
    )
    split_lookup = {split["split"]: split for split in cv_splits}

    for k in k_values:
        for split_number in split_numbers:
            if (k, split_number) in completed:
                continue
            result = evaluate_cv_run(k, split_lookup[split_number])
            results = pd.concat(
                [results, pd.DataFrame([result])], ignore_index=True
            )
            results = results.sort_values(["K", "split"]).reset_index(drop=True)
            results["AUC"] = results["AUC"].round(3)
            results.to_csv(
                cv_results_file, index=False, float_format="%.3f"
            )
            completed.add((k, split_number))

    return results


## Run the complete K-selection grid

Run All evaluates every K from 2 to 15 across all 100 splits. Each completed AUC is immediately checkpointed, so an interrupted run resumes from the remaining K-split combinations.

In [20]:
cv_results = run_cv_grid(
    k_values=candidate_k,
    split_numbers=range(1, n_cv_splits + 1),
)

## Aggregate AUC and select K

Selection uses the K with the highest mean AUC across 100 splits. SD and the curve shape are descriptive and never override that numerical maximum.

In [21]:
if cv_results_file.exists():
    split_auc_results = pd.read_csv(cv_results_file)
    completed_runs = split_auc_results[["K", "split"]].drop_duplicates()
    expected_runs = len(candidate_k) * n_cv_splits

    if len(completed_runs) == expected_runs:
        k_selection = (
            split_auc_results.groupby("K", as_index=False)
            .agg(
                mean_auc=("AUC", "mean"),
                sd_auc=("AUC", "std"),
            )
        )
        selected_k = int(
            k_selection.loc[k_selection["mean_auc"].idxmax(), "K"]
        )
        k_selection["selected"] = k_selection["K"].eq(selected_k)
        k_selection.to_csv(k_selection_file, index=False)

        figure, axis = plt.subplots(figsize=(9, 5))
        axis.errorbar(
            k_selection["K"],
            k_selection["mean_auc"],
            yerr=k_selection["sd_auc"],
            marker="o",
            capsize=3,
        )
        selected_row = k_selection.loc[k_selection["selected"]].iloc[0]
        axis.scatter(
            selected_row["K"],
            selected_row["mean_auc"],
            color="crimson",
            s=70,
            zorder=3,
            label=f"Selected K={selected_k}",
        )
        axis.set(
            title="Hy-MMSBM selection by held-out hyperedge AUC",
            xlabel="Number of communities K",
            ylabel="Mean test AUC (+/- SD)",
            xticks=list(candidate_k),
        )
        axis.legend()
        figure.tight_layout()
        figure.savefig(k_selection_plot_file, dpi=300, bbox_inches="tight")
        plt.close(figure)


4. Fit a full, non-assortative affinity matrix

5. Assess repeated-run stability properly

6. Transform U for interpretation

7.  Define core and mixed-membership diseases, Interpret the affinity matrix W

8.Save membership interpretation part


membership_results = chronic_nodes.copy()

for community in range(U.shape[1]):
    membership_results[f"community_{community + 1}"] = U[:, community]

membership_results.to_csv(
    output_dir / "disease_community_memberships.csv",
    index=False,
)

The model tells :

- which hidden disease communities exist;

- which diseases are the clearest members of each community;

- which diseases belong meaningfully to several communities;

- which disease communities frequently appear together.

- pay attention to model stability and interpretation of membership!

CONSIDER 2LEVEL ANALYSIS OF THE SAME PATIETNS

# Add chronic-disease block analysis

## 1. Create the chronic-group–ICD-block interpretation table

icd blocks 131 of all patients that have at least 2 chronic groups inside